# AI와 함께 타이타닉 데이터 분석 한 사이클 완주하기

STEP 00–17 학생용 실행 Notebook입니다.

핵심 원칙은 **셀을 위에서 아래로 직접 실행하면서 데이터가 어떻게 변하는지 확인하는 것**입니다. 모델링 단계에서도 결측치 처리, 인코딩, 정규화/표준화, 모델 학습을 한꺼번에 묶지 않고 각각 따로 실행합니다.


## STEP 00. 전체 분석 지도

환경 → 데이터 → 품질 → Target → 결측/컬럼/인코딩 원리 → 시각화 → EDA/통계 → Feature → split → 결측치 처리 → 인코딩 → 정규화/표준화 → 모델 학습 → 평가 → 추가 모델 → 최종 선택 → 저장/새 예측 → Streamlit


## STEP 01. 실행 환경 확인


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

print("Python:", sys.executable)
print("Version:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("pandas / numpy / sklearn:", pd.__version__, np.__version__, sklearn.__version__)


Python: /home/gem/research/프로그램 모음/venvs/llm-data-analysis/bin/python
Version: 3.12.3
Working directory: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/titanic_ai_analysis
pandas / numpy / sklearn: 3.0.5 2.5.3 1.9.1


## STEP 02. 데이터 로딩

`data/titanic/train.csv`가 없다면 저장소 루트에서 먼저 `python scripts/prepare_titanic_data.py`를 실행합니다.


In [2]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

data_path = project_root / "data" / "titanic" / "train.csv"
if not data_path.is_file():
    raise FileNotFoundError(
        f"{data_path} 파일이 없습니다. 저장소 루트에서 "
        "python scripts/prepare_titanic_data.py 를 먼저 실행하세요."
    )

df = pd.read_csv(data_path)
display(df.head())
print("shape:", df.shape)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


shape: (891, 12)


## STEP 03. 데이터 구조와 품질 확인


In [3]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_rate": (df.isna().mean() * 100).round(2),
    "nunique": df.nunique(dropna=False),
})
display(quality)
display(df.describe())

for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"\n[{col}]")
    display(df[col].value_counts(dropna=False).to_frame("count"))


,dtype,missing,missing_rate,nunique
PassengerId,int64,0,0.00,891
Survived,int64,0,0.00,2
Pclass,int64,0,0.00,3
Name,str,0,0.00,891
Sex,str,0,0.00,2
Age,float64,177,19.87,89
SibSp,int64,0,0.00,7
Parch,int64,0,0.00,7
Ticket,str,0,0.00,681
Fare,float64,0,0.00,248


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200



[Survived]


,count
Survived,
0,549
1,342



[Pclass]


,count
Pclass,
3,491
1,216
2,184



[Sex]


,count
Sex,
male,577
female,314



[Embarked]


,count
Embarked,
S,644
C,168
Q,77
NaN,2


## STEP 04. Target 정의

`Survived=1`을 Positive class로 두는 이진 분류 문제입니다.


In [4]:
print(df["Survived"].value_counts(dropna=False).sort_index())
print({"task": "binary_classification", "target": "Survived", "positive_class": 1})


Survived
0    549
1    342
Name: count, dtype: int64
{'task': 'binary_classification', 'target': 'Survived', 'positive_class': 1}


## STEP 05. 결측치 처리 원리 학습

여기서 만드는 `df_work`는 EDA용입니다. 모델링에서는 다시 원본 `df`에서 시작합니다.


In [5]:
df_work = df.copy()

age_median_for_eda = df_work["Age"].median()
embarked_mode_for_eda = df_work["Embarked"].mode(dropna=True).iloc[0]

print("EDA Age median:", age_median_for_eda)
print("EDA Embarked mode:", embarked_mode_for_eda)

df_work["Age"] = df_work["Age"].fillna(age_median_for_eda)
df_work["Embarked"] = df_work["Embarked"].fillna(embarked_mode_for_eda)

display(df_work[["Age", "Embarked", "Cabin"]].isna().sum().to_frame("missing"))


EDA Age median: 28.0
EDA Embarked mode: S


,missing
Age,0
Embarked,0
Cabin,687


## STEP 06. 컬럼 사용 정책 검토


In [6]:
candidate_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
for col in candidate_cols:
    print(f"\n[{col}] dtype={df_work[col].dtype}, "
          f"nunique={df_work[col].nunique(dropna=False)}, "
          f"missing={df_work[col].isna().sum()}")
    print(df_work[col].dropna().astype(str).head(5).tolist())



[PassengerId] dtype=int64, nunique=891, missing=0
['1', '2', '3', '4', '5']

[Name] dtype=str, nunique=891, missing=0
['Braund, Mr. Owen Harris', 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'Heikkinen, Miss Laina', 'Futrelle, Mrs. Jacques Heath (Lily May Peel)', 'Allen, Mr. William Henry']

[Ticket] dtype=str, nunique=681, missing=0
['A/5 21171', 'PC 17599', 'STON/O2. 3101282', '113803', '373450']

[Cabin] dtype=str, nunique=148, missing=687
['C85', 'C123', 'E46', 'G6', 'C103']


## STEP 07. 범주형 인코딩 원리 학습

`df_encoded`는 인코딩 원리를 보기 위한 연습용이며 모델 입력으로 사용하지 않습니다.


In [7]:
df_encoded = df_work.copy()
encoded_preview = pd.get_dummies(
    df_encoded[["Sex", "Embarked"]],
    columns=["Sex", "Embarked"],
    dtype=int,
)
display(encoded_preview.head())


,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0,1,0,0,1
1,1,0,1,0,0
2,1,0,0,0,1
3,1,0,0,0,1
4,0,1,0,0,1


## STEP 08. 시각화


In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df_work, x="Survived")
plt.title("Survived distribution")
plt.show()

sns.barplot(data=df_work, x="Sex", y="Survived")
plt.title("Survival rate by Sex")
plt.show()

sns.barplot(data=df_work, x="Pclass", y="Survived")
plt.title("Survival rate by Pclass")
plt.show()

sns.boxplot(data=df_work, x="Survived", y="Fare")
plt.title("Fare by Survived")
plt.show()


## STEP 09. 기초 통계와 EDA


In [9]:
sex_summary = (
    df_work.groupby("Sex", dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)
pclass_summary = (
    df_work.groupby("Pclass", dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)
sex_pclass_summary = (
    df_work.groupby(["Sex", "Pclass"], dropna=False)
    .agg(rows=("Survived", "size"), survival_rate=("Survived", "mean"))
)

display(sex_summary)
display(pclass_summary)
display(sex_pclass_summary)


,rows,survival_rate
Sex,,
female,314,0.742038
male,577,0.188908


,rows,survival_rate
Pclass,,
1,216,0.629630
2,184,0.472826
3,491,0.242363


rows  survival_rate
Sex    Pclass                     
female 1         94       0.968085
       2         76       0.921053
       3        144       0.500000
male   1        122       0.368852
       2        108       0.157407
       3        347       0.135447

### STEP 09-A. Fare 평균 차이 — Welch 독립표본 t-test

- H0: 생존자와 비생존자의 평균 Fare에는 차이가 없다.
- H1: 두 그룹의 평균 Fare에는 차이가 있다.
- `p-value < 0.05`이면 보통 H0를 기각합니다.


In [10]:
from scipy.stats import ttest_ind

survived_fare = df_work.loc[df_work["Survived"] == 1, "Fare"].dropna()
not_survived_fare = df_work.loc[df_work["Survived"] == 0, "Fare"].dropna()

stat, p_value = ttest_ind(survived_fare, not_survived_fare, equal_var=False)

print("survived Fare mean:", round(survived_fare.mean(), 2), " n =", len(survived_fare))
print("not survived Fare mean:", round(not_survived_fare.mean(), 2), " n =", len(not_survived_fare))
print("t-statistic:", round(stat, 4))
print(f"p-value: {p_value:.3e}")


survived Fare mean: 48.4  n = 342
not survived Fare mean: 22.12  n = 549
t-statistic: 6.8391
p-value: 2.699e-11


## STEP 10. Feature 설계

이번 기본 실행에서는 `FamilySize`, `IsAlone`을 직접 만들어 봅니다.


In [11]:
feature_demo = df_work.copy()
feature_demo["FamilySize"] = feature_demo["SibSp"] + feature_demo["Parch"] + 1
feature_demo["IsAlone"] = (feature_demo["FamilySize"] == 1).astype(int)

display(feature_demo[["SibSp", "Parch", "FamilySize", "IsAlone", "Survived"]].head())


,SibSp,Parch,FamilySize,IsAlone,Survived
0,1,0,2,0,0
1,1,0,2,0,1
2,0,0,1,1,1
3,1,0,2,0,1
4,0,0,1,1,0


# Part 2. 모델링 — STEP 11–15

이제 `df_work`, `df_encoded`, `feature_demo`가 아니라 **원본 `df`에서 다시 시작**합니다.


## STEP 11. 학습/테스트 데이터 준비

Feature 생성부터 split까지 직접 실행합니다.


In [12]:
model_source = df.copy()
model_source["FamilySize"] = model_source["SibSp"] + model_source["Parch"] + 1
model_source["IsAlone"] = (model_source["FamilySize"] == 1).astype(int)

raw_input_columns = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
feature_columns = raw_input_columns + ["FamilySize", "IsAlone"]
numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked", "IsAlone"]

X = model_source[feature_columns].copy()
y = model_source["Survived"].astype(int).copy()

print("X:", X.shape)
print("y:", y.shape)
display(X.head())


X: (891, 9)
y: (891,)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone
0,3,male,22.0,1,0,7.2500,S,2,0
1,1,female,38.0,1,0,71.2833,C,2,0
2,3,female,26.0,0,0,7.9250,S,1,1
3,1,female,35.0,1,0,53.1000,S,2,0
4,3,male,35.0,0,0,8.0500,S,1,1


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("train:", X_train.shape, y_train.shape)
print("test :", X_test.shape, y_test.shape)

print("전체 비율")
display(y.value_counts(normalize=True).sort_index().rename("ratio").to_frame())
print("학습 비율")
display(y_train.value_counts(normalize=True).sort_index().rename("train_ratio").to_frame())
print("테스트 비율")
display(y_test.value_counts(normalize=True).sort_index().rename("test_ratio").to_frame())


train: (712, 9) (712,)
test : (179, 9) (179,)
전체 비율


,ratio
Survived,
0,0.616162
1,0.383838


학습 비율


,train_ratio
Survived,
0,0.616573
1,0.383427


테스트 비율


,test_ratio
Survived,
0,0.614525
1,0.385475


## STEP 12. 전처리를 하나씩 직접 실행하기

이번 STEP에서는 한꺼번에 묶지 않습니다.

1. 숫자형 결측치 처리
2. 범주형 결측치 처리
3. One-Hot Encoding
4. 정규화와 표준화 차이 확인
5. StandardScaler로 표준화
6. 숫자형과 범주형 데이터 결합
7. Logistic Regression 학습


### STEP 12-1. 숫자형 결측치 처리

중앙값은 **Train에서만 계산**하고 Test에는 같은 값을 적용합니다.


In [14]:
from sklearn.impute import SimpleImputer

numeric_imputer = SimpleImputer(strategy="median")

X_train_num_imputed = pd.DataFrame(
    numeric_imputer.fit_transform(X_train[numeric_features]),
    columns=numeric_features,
    index=X_train.index,
)
X_test_num_imputed = pd.DataFrame(
    numeric_imputer.transform(X_test[numeric_features]),
    columns=numeric_features,
    index=X_test.index,
)

print("Train numeric missing:", X_train_num_imputed.isna().sum().sum())
print("Test numeric missing :", X_test_num_imputed.isna().sum().sum())
display(X_train_num_imputed.head())


Train numeric missing: 0
Test numeric missing : 0


,Age,SibSp,Parch,Fare,FamilySize
692,28.5,0.0,0.0,56.4958,1.0
481,28.5,0.0,0.0,0.0000,1.0
527,28.5,0.0,0.0,221.7792,1.0
855,18.0,0.0,1.0,9.3500,2.0
801,31.0,1.0,1.0,26.2500,3.0


### STEP 12-2. 범주형 결측치 처리

최빈값도 **Train에서만 학습**합니다.


In [15]:
categorical_imputer = SimpleImputer(strategy="most_frequent")

X_train_cat_imputed = pd.DataFrame(
    categorical_imputer.fit_transform(X_train[categorical_features]),
    columns=categorical_features,
    index=X_train.index,
)
X_test_cat_imputed = pd.DataFrame(
    categorical_imputer.transform(X_test[categorical_features]),
    columns=categorical_features,
    index=X_test.index,
)

print("Train categorical missing:", X_train_cat_imputed.isna().sum().sum())
print("Test categorical missing :", X_test_cat_imputed.isna().sum().sum())
display(X_train_cat_imputed.head())


Train categorical missing: 0
Test categorical missing : 0


,Pclass,Sex,Embarked,IsAlone
692,3,male,S,1
481,2,male,S,1
527,1,male,S,1
855,3,female,S,0
801,2,female,S,0


### STEP 12-3. One-Hot Encoding

범주 목록은 Train에서 학습하고 Test에는 같은 열 구조를 적용합니다.


In [16]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat_encoded_array = encoder.fit_transform(X_train_cat_imputed)
X_test_cat_encoded_array = encoder.transform(X_test_cat_imputed)

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded_array,
    columns=encoded_feature_names,
    index=X_train.index,
)
X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded_array,
    columns=encoded_feature_names,
    index=X_test.index,
)

print("encoded columns:", list(encoded_feature_names))
display(X_train_cat_encoded.head())


encoded columns: ['Pclass_1', 'Pclass_2', 'Pclass_3', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'IsAlone_0', 'IsAlone_1']


,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,IsAlone_0,IsAlone_1
692,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
481,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
527,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
855,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
801,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


### STEP 12-4. 정규화와 표준화 구분

- **정규화(Normalization)**: 값을 일정 범위로 맞춤. 대표적으로 `MinMaxScaler`를 사용하면 보통 0~1 범위가 됩니다.
- **표준화(Standardization)**: 평균 0, 표준편차 1을 기준으로 값을 바꿉니다. 대표적으로 `StandardScaler`를 사용합니다.

이번 Logistic Regression 실습에서는 **StandardScaler를 이용한 표준화**를 실제 모델 입력에 사용합니다. 먼저 Min-Max 정규화가 어떻게 보이는지만 간단히 확인합니다.


In [17]:
from sklearn.preprocessing import MinMaxScaler

minmax_demo = MinMaxScaler()
minmax_preview = pd.DataFrame(
    minmax_demo.fit_transform(X_train_num_imputed[["Age", "Fare"]]),
    columns=["Age_minmax", "Fare_minmax"],
    index=X_train.index,
)

display(minmax_preview.head())
print("이 결과는 개념 확인용이며 실제 모델 입력에는 사용하지 않습니다.")


,Age_minmax,Fare_minmax
692,0.352852,0.110272
481,0.352852,0.000000
527,0.352852,0.432884
855,0.220910,0.018250
801,0.384267,0.051237


이 결과는 개념 확인용이며 실제 모델 입력에는 사용하지 않습니다.


### STEP 12-5. StandardScaler로 표준화

`fit_transform()`은 Train에, `transform()`은 Test에 사용합니다.


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_num_imputed),
    columns=numeric_features,
    index=X_train.index,
)
X_test_num_scaled = pd.DataFrame(
    scaler.transform(X_test_num_imputed),
    columns=numeric_features,
    index=X_test.index,
)

display(X_train_num_scaled.head())
display(X_train_num_scaled.describe().loc[["mean", "std"]])


,Age,SibSp,Parch,Fare,FamilySize
692,-0.081135,-0.465084,-0.466183,0.513812,-0.556339
481,-0.081135,-0.465084,-0.466183,-0.662563,-0.556339
527,-0.081135,-0.465084,-0.466183,3.955399,-0.556339
855,-0.887827,-0.465084,0.727782,-0.467874,0.073412
801,0.110934,0.478335,0.727782,-0.115977,0.703162


,Age,SibSp,Parch,Fare,FamilySize
mean,7.734138e-17,-5.613487e-18,-1.621674e-17,-1.746418e-17,3.492836e-17
std,1.000703e+00,1.000703e+00,1.000703e+00,1.000703e+00,1.000703e+00


### STEP 12-6. 숫자형과 범주형 데이터 결합


In [19]:
X_train_ready = pd.concat(
    [X_train_num_scaled, X_train_cat_encoded], axis=1
)
X_test_ready = pd.concat(
    [X_test_num_scaled, X_test_cat_encoded], axis=1
)

print("X_train_ready:", X_train_ready.shape)
print("X_test_ready :", X_test_ready.shape)
print("missing train:", X_train_ready.isna().sum().sum())
print("missing test :", X_test_ready.isna().sum().sum())
display(X_train_ready.head())


X_train_ready: (712, 15)
X_test_ready : (179, 15)
missing train: 0
missing test : 0


,Age,SibSp,Parch,Fare,FamilySize,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,IsAlone_0,IsAlone_1
692,-0.081135,-0.465084,-0.466183,0.513812,-0.556339,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
481,-0.081135,-0.465084,-0.466183,-0.662563,-0.556339,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
527,-0.081135,-0.465084,-0.466183,3.955399,-0.556339,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
855,-0.887827,-0.465084,0.727782,-0.467874,0.073412,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
801,0.110934,0.478335,0.727782,-0.115977,0.703162,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0


### STEP 12-7. Logistic Regression Baseline 학습


In [20]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_ready, y_train)

print("model:", baseline_model)
print("classes:", baseline_model.classes_)


model: LogisticRegression(max_iter=1000, random_state=42)
classes: [0 1]


## STEP 13. 성능 평가와 Confusion Matrix


In [21]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

baseline_pred = baseline_model.predict(X_test_ready)

print("accuracy :", round(accuracy_score(y_test, baseline_pred), 4))
print("precision:", round(precision_score(y_test, baseline_pred), 4))
print("recall   :", round(recall_score(y_test, baseline_pred), 4))
print("f1       :", round(f1_score(y_test, baseline_pred), 4))
print()
print(classification_report(y_test, baseline_pred, digits=4))

cm = confusion_matrix(y_test, baseline_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
print("Confusion Matrix:\n", cm)
print({"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)})


accuracy : 0.8156
precision: 0.8103
recall   : 0.6812
f1       : 0.7402



              precision    recall  f1-score   support

           0     0.8182    0.9000    0.8571       110
           1     0.8103    0.6812    0.7402        69

    accuracy                         0.8156       179
   macro avg     0.8143    0.7906    0.7987       179
weighted avg     0.8152    0.8156    0.8120       179

Confusion Matrix:
 [[99 11]
 [22 47]]
{'TN': 99, 'FP': 11, 'FN': 22, 'TP': 47}


Confusion Matrix는 단순히 몇 개를 맞혔는지만 보는 것이 아니라 **어떤 종류의 실수(FP/FN)를 했는지** 보는 표입니다. 암 진단에서는 FN이 특히 중요할 수 있고, 스팸 분류에서는 정상 메일을 스팸으로 보내는 FP가 중요할 수 있습니다.


## STEP 14. 추가 모델 — Random Forest

Random Forest는 표준화가 필수는 아니지만, 이번에는 모델 비교를 단순하게 하기 위해 같은 준비 데이터를 사용합니다.


In [22]:
from sklearn.ensemble import RandomForestClassifier

additional_model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)
additional_model.fit(X_train_ready, y_train)
additional_pred = additional_model.predict(X_test_ready)

print("Random Forest accuracy:", round(accuracy_score(y_test, additional_pred), 4))
print("Random Forest f1      :", round(f1_score(y_test, additional_pred), 4))


Random Forest accuracy: 0.8101
Random Forest f1      : 0.7258


## STEP 15. 모델 비교와 최종 모델 선택

이번 기본 실습에서는 같은 Test set의 결과를 비교해 최종 모델을 선택합니다. 더 엄밀한 실무에서는 별도 Validation set이나 Cross Validation을 추가할 수 있습니다.


In [23]:
model_compare = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_test, baseline_pred),
        "f1": f1_score(y_test, baseline_pred),
    },
    {
        "model": "Random Forest",
        "accuracy": accuracy_score(y_test, additional_pred),
        "f1": f1_score(y_test, additional_pred),
    },
])

display(model_compare)

FINAL_MODEL_CHOICE = "baseline"

if FINAL_MODEL_CHOICE == "baseline":
    final_model = baseline_model
    final_model_name = "Logistic Regression"
elif FINAL_MODEL_CHOICE == "random_forest":
    final_model = additional_model
    final_model_name = "Random Forest"
else:
    raise ValueError("FINAL_MODEL_CHOICE는 baseline 또는 random_forest여야 합니다.")

print("final model:", final_model_name)


,model,accuracy,f1
0,Logistic Regression,0.815642,0.740157
1,Random Forest,0.810056,0.725806


final model: Logistic Regression


# Part 3. 저장·새 입력 예측·서비스


## STEP 16. 전처리 객체와 모델 저장

한 개의 Pipeline을 저장하지 않고, 지금까지 직접 학습한 객체들을 각각 묶어 저장합니다.


In [24]:
import json
import joblib

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

bundle_path = models_dir / "titanic_model_bundle.joblib"
contract_path = models_dir / "titanic_model_contract.json"

model_bundle = {
    "numeric_imputer": numeric_imputer,
    "categorical_imputer": categorical_imputer,
    "encoder": encoder,
    "scaler": scaler,
    "model": final_model,
}
joblib.dump(model_bundle, bundle_path)

model_contract = {
    "task": "binary_classification",
    "target": "Survived",
    "positive_class": 1,
    "raw_input_columns": raw_input_columns,
    "model_feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "prepared_feature_columns": X_train_ready.columns.tolist(),
    "derived_features": ["FamilySize", "IsAlone"],
    "scaling": "StandardScaler",
    "final_estimator": type(final_model).__name__,
}
contract_path.write_text(
    json.dumps(model_contract, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("saved bundle  :", bundle_path)
print("saved contract:", contract_path)


saved bundle  : /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/titanic_ai_analysis/models/titanic_model_bundle.joblib
saved contract: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/titanic_ai_analysis/models/titanic_model_contract.json


### STEP 16-A. 새로운 승객을 같은 순서로 직접 변환하고 예측


In [25]:
loaded_bundle = joblib.load(bundle_path)

new_passenger = pd.DataFrame([{
    "Pclass": 3,
    "Sex": "male",
    "Age": 30.0,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 10.0,
    "Embarked": "S",
}])

new_passenger["FamilySize"] = new_passenger["SibSp"] + new_passenger["Parch"] + 1
new_passenger["IsAlone"] = (new_passenger["FamilySize"] == 1).astype(int)

new_num_imputed = pd.DataFrame(
    loaded_bundle["numeric_imputer"].transform(new_passenger[numeric_features]),
    columns=numeric_features,
)
new_num_scaled = loaded_bundle["scaler"].transform(new_num_imputed)

new_cat_imputed = pd.DataFrame(
    loaded_bundle["categorical_imputer"].transform(new_passenger[categorical_features]),
    columns=categorical_features,
)
new_cat_encoded = loaded_bundle["encoder"].transform(new_cat_imputed)

new_ready = pd.DataFrame(
    np.hstack([new_num_scaled, new_cat_encoded]),
    columns=model_contract["prepared_feature_columns"],
)

display(new_ready)

loaded_model = loaded_bundle["model"]
new_prediction = int(loaded_model.predict(new_ready)[0])
positive_index = list(loaded_model.classes_).index(1)
positive_probability = float(loaded_model.predict_proba(new_ready)[0][positive_index])

print("prediction:", new_prediction)
print("survival probability:", round(positive_probability, 4))


,Age,SibSp,Parch,Fare,FamilySize,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,IsAlone_0,IsAlone_1
0,0.034106,-0.465084,-0.466183,-0.45434,-0.556339,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


prediction: 0
survival probability: 0.0813


## STEP 17. Streamlit 서비스

Notebook에서 직접 확인한 순서를 앱에서도 그대로 사용합니다.

`원본 입력 → FamilySize/IsAlone → 결측치 처리 → One-Hot Encoding → StandardScaler → 모델 예측`

실행:

```powershell
streamlit run src/titanic_app/app.py
```

기본 입력으로 폼을 제출한 결과 class 0(비생존), 생존 확률 8.1%가 표시됐고 앱 테스트와 HTTP health check를 통과했습니다.

![Streamlit 실행 결과](images/step17_streamlit.png)


# 최종 정리

- STEP 07 인코딩은 원리 학습용입니다.
- STEP 11에서 원본 `df`로 다시 시작하고 먼저 Train/Test를 나눕니다.
- STEP 12에서 결측치 처리, One-Hot Encoding, 정규화/표준화를 각각 직접 실행합니다.
- `MinMaxScaler`는 정규화 개념 확인용이고 실제 기본 모델에는 `StandardScaler` 표준화를 사용합니다.
- Train에는 `fit_transform()`, Test에는 `transform()`을 사용해 데이터 누수를 막습니다.
- STEP 16에서는 전처리 객체와 모델을 따로 저장하고 새 승객에게 같은 순서로 적용합니다.


## 실습 기록 요약

- 실행일: 2026-09-29
- GitHub: <https://github.com/chocoemong1/llm-data-analysis-study/blob/main/titanic_ai_analysis/titanic_ai_analysis.ipynb>

| 구간 | 관찰 | 해석·판단 | 한계 |
| --- | --- | --- | --- |
| 데이터 | 891행×12열이며 Age 177건, Cabin 687건, Embarked 2건이 결측이다. 생존 342명, 비생존 549명이다. | EDA에서는 Age 중앙값 28, Embarked 최빈값 S로 채우되 모델링에서는 원본에서 다시 시작한다. | 결측 발생 원인은 확인할 수 없다. |
| EDA | 여성 생존율 0.742, 남성 0.189이고 1등석 0.630, 3등석 0.242였다. 생존자의 평균 Fare는 48.40, 비생존자는 22.12이며 Welch 검정 p=2.699e-11이다. | 그룹 차이는 모델 후보 Feature를 정하는 근거로 사용한다. | 관찰·통계적 차이는 성별·등급·운임이 생존의 원인임을 증명하지 않는다. |
| 전처리 | 712/179로 층화 분할했고 최종 입력은 15열, Train/Test 결측은 모두 0이다. | Train에만 `fit_transform()`, Test에는 `transform()`을 적용했다. Min-Max는 개념 확인용, 실제 모델에는 StandardScaler를 사용했다. | 단일 holdout 결과라 분할에 따른 변동이 있다. |
| 모델 | Logistic Regression accuracy 0.8156, F1 0.7402; Random Forest accuracy 0.8101, F1 0.7258이다. | 두 지표가 모두 더 높은 Logistic Regression을 최종 선택했다. 더 복잡한 모델이라는 이유만으로 Random Forest를 택하지 않았다. | 별도 validation과 교차검증·튜닝을 하지 않았다. |
| 새 입력 | 예시 승객은 class 0, 생존 확률 0.0813으로 예측됐다. | 저장된 전처리 객체와 모델을 학습 때와 같은 순서로 재사용했다. | 수업용 데이터의 모델 출력이며 개인의 실제 생존 가능성을 뜻하지 않는다. |

**AI 제안 검토:** 전체 과정을 한 번에 묶지 않고 단계별 출력으로 검증했다. 새 입력 처리에서 feature-name 경고가 나오는 제안은 imputed 배열을 동일한 컬럼명의 DataFrame으로 복원하도록 수정했다.

**다음 단계:** 교차검증과 threshold 비교를 추가하고, 반복 전처리 코드를 검증 가능한 함수로 정리한다.
